<a href="https://colab.research.google.com/github/adhithyannnn/machineLearning/blob/main/exp12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [7]:
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

In [8]:
features = [
    "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]

In [9]:
X = df[features]
y = df["Survived"]

In [10]:
numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

In [11]:
categorical_features = ["Sex", "Embarked"]

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

In [14]:
base_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

In [15]:
bagging = BaggingClassifier(
    estimator=base_model,
    n_estimators=50,
    random_state=42
)

In [16]:
bagging_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", bagging)
])

In [17]:
bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)


In [18]:
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=2,
        random_state=42
    ),
    n_estimators=50,
    random_state=42
)

In [19]:
boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", boosting)
])

In [22]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Re-define numerical transformer with an imputer
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Re-define categorical transformer with an imputer and encoder
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Re-create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numeric_features),
        ("cat", cat_transformer, categorical_features)
    ]
)

# Re-build pipelines with the updated preprocessor
bagging_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", bagging)
])

boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", boosting)
])

# Fit and predict
boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

In [23]:
def evaluate_model(name, y_true, y_pred):
    print("\n", name)

In [26]:
def evaluate_model(name, y_true, y_pred):
    print("\n", name)
    print("Accuracy: ", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall:   ", round(recall_score(y_true, y_pred), 4))
    print("F1-score: ", round(f1_score(y_true, y_pred), 4))

evaluate_model("Bagging Classifier", y_test, bag_pred)
evaluate_model("AdaBoost Classifier", y_test, boost_pred)


 Bagging Classifier
Accuracy:  0.7989
Precision: 0.8667
Recall:    0.5652
F1-score:  0.6842

 AdaBoost Classifier
Accuracy:  0.7765
Precision: 0.7636
Recall:    0.6087
F1-score:  0.6774


In [27]:

evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)


 Bagging
Accuracy:  0.7989
Precision: 0.8667
Recall:    0.5652
F1-score:  0.6842

 Boosting (AdaBoost)
Accuracy:  0.7765
Precision: 0.7636
Recall:    0.6087
F1-score:  0.6774
